# 🚀 项目 1: 手写 ReAct Agent — GitHub 仓库分析

> 从零构建生产级 Agent，真实 GitHub API 调用，逐步拆解每一层。

## 架构全景

```
┌─────────────────────────────────────────────────────┐
│                  Agent Loop (ReAct)                 │
│  感知 → 决策 → 执行 → 反馈 → 再决策...               │
├─────────────────────────────────────────────────────┤
│  弹性保护层                                         │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐         │
│  │  重试层   │→ │ 熔断层   │→ │ 降级层   │          │
│  │ 指数退避  │  │ 三态机   │  │ 优雅错误  │          │
│  └──────────┘  └──────────┘  └──────────┘         │
├─────────────────────────────────────────────────────┤
│  工具层                                             │
│  get_repo_info │ list_directory │ read_file │ commits │
├─────────────────────────────────────────────────────┤
│  HTTP 层 (httpx + GitHub REST API v3)               │
└─────────────────────────────────────────────────────┘
```

## 第 1 步：环境准备

In [1]:
import asyncio, json, os, random, sys, time
import httpx
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
print("✅ 依赖加载完成")
print(f"   DeepSeek API: {'已配置' if os.getenv('API_KEY') else '❌ 未配置'}")
print(f"   GitHub Token: {'已配置' if os.getenv('GITHUB_TOKEN') else '⚠️ 未配置（60次/小时限制）'}")

✅ 依赖加载完成
   DeepSeek API: 已配置
   GitHub Token: 已配置


## 第 2 步：熔断器 — 三态状态机

面试常考！CLOSED → OPEN → HALF_OPEN 三态转换：

In [2]:
class CircuitBreaker:
    """熔断器 — 防止对故障服务持续发起无效请求。"""

    def __init__(self, name, threshold=5, reset_timeout=30):
        self.name = name
        self.threshold = threshold
        self.reset_timeout = reset_timeout
        self.failure_count = 0
        self.last_failure_time = 0.0
        self.state = "CLOSED"

    def allow_request(self):
        if self.state == "CLOSED":
            return True
        if self.state == "OPEN":
            if time.time() - self.last_failure_time >= self.reset_timeout:
                self.state = "HALF_OPEN"  # 探测窗口
                return True
            return False
        return True  # HALF_OPEN: 允许一次探测

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.threshold:
            self.state = "OPEN"

# 模拟熔断
cb = CircuitBreaker("demo", threshold=3)
print(f"初始状态: {cb.state}")
for i in range(3):
    cb.record_failure()
    print(f"  第{i+1}次失败后: {cb.state}")
print(f"允许请求? {cb.allow_request()}")  # False — 已熔断

初始状态: CLOSED
  第1次失败后: CLOSED
  第2次失败后: CLOSED
  第3次失败后: OPEN
允许请求? False


## 第 3 步：GitHub API 客户端（带弹性保护）

In [3]:
GITHUB_API_BASE = "https://api.github.com"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

RESILIENCE_CONFIG = {
    "max_retries": 3,
    "base_delay": 0.5,
    "max_delay": 8.0,
    "request_timeout": 15.0,
}

_breakers = {}  # 工具 → CircuitBreaker 映射


async def github_request(tool_name, url, verbose=True):
    """发送 GitHub API 请求，带重试 + 熔断 + 降级。"""
    if tool_name not in _breakers:
        _breakers[tool_name] = CircuitBreaker(tool_name, threshold=5)
    cb = _breakers[tool_name]

    if not cb.allow_request():
        return {"ok": False, "error": "熔断器开路"}

    headers = {"Accept": "application/vnd.github.v3+json", "User-Agent": "react-agent/1.0"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

    async with httpx.AsyncClient(follow_redirects=True) as client:
        for attempt in range(RESILIENCE_CONFIG["max_retries"] + 1):
            try:
                resp = await client.get(url, headers=headers, timeout=15.0)
                if resp.status_code == 200:
                    cb.record_success()
                    return {"ok": True, "data": resp.json()}
                if resp.status_code == 404:
                    cb.record_success()
                    return {"ok": False, "error": "资源不存在"}
                if resp.status_code in (429, 500, 502, 503, 504):
                    cb.record_failure()
                    wait = min(0.5 * (2**attempt), 8.0)
                    if verbose:
                        print(f"  ⏳ HTTP {resp.status_code}, 重试({attempt+1}), 等待 {wait:.1f}s")
                    await asyncio.sleep(wait)
                    continue
            except Exception as e:
                cb.record_failure()
                wait = min(0.5 * (2**attempt), 8.0)
                if verbose:
                    print(f"  ⚡ {type(e).__name__}, 重试({attempt+1}), 等待 {wait:.1f}s")
                await asyncio.sleep(wait)

    return {"ok": False, "error": "全部重试失败", "_fallback": True}

print("✅ GitHub API 客户端就绪")

✅ GitHub API 客户端就绪


## 第 4 步：工具实现 — 4 个 GitHub 分析工具

In [4]:
async def get_repo_info(owner, repo, verbose=True):
    result = await github_request("repo_info", f"{GITHUB_API_BASE}/repos/{owner}/{repo}", verbose)
    if not result.get("ok"):
        return result
    d = result["data"]
    return {"ok": True, "name": d.get("full_name"), "stars": d.get("stargazers_count"),
            "forks": d.get("forks_count"), "language": d.get("language"),
            "description": d.get("description"), "topics": d.get("topics", []),
            "license": (d.get("license") or {}).get("spdx_id"),
            "created_at": d.get("created_at"), "updated_at": d.get("updated_at")}

async def list_directory(owner, repo, path="", verbose=True):
    url = f"{GITHUB_API_BASE}/repos/{owner}/{repo}/contents/{path}".rstrip("/")
    result = await github_request("list_dir", url, verbose)
    if not result.get("ok"):
        return result
    data = result["data"]
    if isinstance(data, list):
        items = [{"name": i["name"], "type": i["type"]} for i in data]
        return {"ok": True, "path": path or "/", "items": items}
    if isinstance(data, dict):
        return {"ok": True, "path": path or "/", "type": "file", "name": data["name"]}
    return result

async def read_file(owner, repo, path, verbose=True):
    import base64
    result = await github_request("read_file", f"{GITHUB_API_BASE}/repos/{owner}/{repo}/contents/{path}", verbose)
    if not result.get("ok") or isinstance(result.get("data"), list):
        return {"ok": False, "error": "不是有效的文件路径"}
    d = result["data"]
    if d.get("encoding") == "base64":
        content = base64.b64decode(d["content"]).decode("utf-8", errors="replace")
    else:
        content = d.get("content", "")
    truncated = len(content) > 4000
    return {"ok": True, "path": path, "size": d.get("size"),
            "content": content[:4000], "truncated": truncated}

async def get_commits(owner, repo, per_page=10, verbose=True):
    result = await github_request("commits", f"{GITHUB_API_BASE}/repos/{owner}/{repo}/commits?per_page={min(per_page,30)}", verbose)
    if not result.get("ok"):
        return result
    commits = []
    for c in result["data"]:
        commits.append({"sha": c["sha"][:8], "author": (c.get("commit",{}).get("author",{}) or {}).get("name","?"),
                         "message": c.get("commit",{}).get("message","").split("\n")[0][:80],
                         "date": (c.get("commit",{}).get("author",{}) or {}).get("date","")})
    return {"ok": True, "commits": commits, "count": len(commits)}

print("✅ 4 个工具已就绪: get_repo_info | list_directory | read_file | get_commits")

✅ 4 个工具已就绪: get_repo_info | list_directory | read_file | get_commits


## 第 5 步：Tool Schema 定义（OpenAI Function Calling 格式）

这是 LLM 理解工具能力的"说明书"。每个 schema 三要素：name + description + parameters(JSON Schema)。

In [5]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_repo_info",
            "description": "获取 GitHub 仓库基本信息：星标、Fork、语言、描述、许可证等。作为分析第一步。",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "仓库拥有者，如 'tiangolo'"},
                    "repo": {"type": "string", "description": "仓库名称，如 'fastapi'"},
                },
                "required": ["owner", "repo"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": "列出仓库目录结构。用于了解项目文件组织。",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "仓库拥有者"},
                    "repo": {"type": "string", "description": "仓库名称"},
                    "path": {"type": "string", "description": "目录路径，空字符串表示根目录"},
                },
                "required": ["owner", "repo"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取仓库中的文件内容。用于阅读 README、配置文件、源码等。最长 4000 字符。",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "仓库拥有者"},
                    "repo": {"type": "string", "description": "仓库名称"},
                    "path": {"type": "string", "description": "文件路径，如 'README.md'"},
                },
                "required": ["owner", "repo", "path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_commits",
            "description": "获取仓库最近提交记录。用于评估开发活跃度。",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "仓库拥有者"},
                    "repo": {"type": "string", "description": "仓库名称"},
                    "per_page": {"type": "integer", "description": "返回数量，默认10，最大30"},
                },
                "required": ["owner", "repo"],
            },
        },
    },
]

TOOL_EXECUTORS = {
    "get_repo_info": get_repo_info,
    "list_directory": list_directory,
    "read_file": read_file,
    "get_commits": get_commits,
}

print("✅ Tool Schema 定义完成，共 4 个工具")

✅ Tool Schema 定义完成，共 4 个工具


## 第 6 步：Agent 循环 — ReAct 核心

这是整个 Agent 的"大脑回路"。循环执行：

1. 发消息历史 + 工具列表给 LLM
2. LLM 返回 text → 这是最终答案，退出
3. LLM 返回 tool_calls → 执行工具 → 追加结果到历史 → 回到第 1 步

In [6]:
SYSTEM_PROMPT = """你是一个专业的 GitHub 仓库分析师。工具：get_repo_info / list_directory / read_file / get_commits。

标准分析流程：
1. get_repo_info → 了解仓库概况
2. list_directory → 浏览项目结构（根目录 + 关键子目录）
3. read_file → 读 README 和关键配置文件
4. get_commits → 了解开发活跃度
5. 输出结构化 Markdown 分析报告（项目概况 / 技术栈 / 目录结构 / 活跃度 / 亮点与建议）

如工具返回错误，如实说明，不编造数据。报告信息密度高，避免空洞评价。"""


async def run_agent(llm_client, user_query, model="deepseek-v4-flash", max_turns=12, verbose=True):
    """ReAct Agent 主循环。"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for turn in range(1, max_turns + 1):
        response = await llm_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.0,
        )

        msg = response.choices[0].message

        if msg.tool_calls:
            names = [tc.function.name for tc in msg.tool_calls]
            if verbose:
                # 打印 CoT（模型对当前步骤的思考）
                if msg.content:
                    print(f"\n  [轮次 {turn}] 💭 {msg.content}")
                # 打印 reasoning_content（DeepSeek 内部推理链）
                reasoning = getattr(msg, "reasoning_content", None)
                if reasoning:
                    print(f"           🧠 {reasoning[:200]}")
                print(f"           🎯 → {', '.join(names)}")
            messages.append(msg.model_dump())

            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                if verbose:
                    print(f"           {name}({json.dumps(args, ensure_ascii=False)})")

                func = TOOL_EXECUTORS.get(name)
                try:
                    result = await func(**args, verbose=verbose)
                except Exception as e:
                    result = {"ok": False, "error": str(e)}

                result_json = json.dumps(result, ensure_ascii=False)
                if verbose:
                    preview = result_json[:120] + "..." if len(result_json) > 120 else result_json
                    print(f"           ← {preview}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result_json,
                })
        else:
            if verbose:
                print(f"\n  [轮次 {turn}] ✅ Done")
            return msg.content

    return "分析超时，请缩小分析范围。"

print("✅ Agent 循环就绪")

✅ Agent 循环就绪


## 第 7 步：实战测试

选择你想分析的 GitHub 仓库，Agent 会自动探索并生成报告。

In [7]:
llm_client = AsyncOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepseek.com",
)

# 🔧 修改这里的 GitHub URL，分析你感兴趣的仓库
QUERY = "请全面分析 GitHub 仓库 https://github.com/TallMessiWu/personal-website，包括项目概况、技术栈、目录结构、开发活跃度，并给出亮点与改进建议"

print(f"👤 用户: {QUERY}\n")
answer = await run_agent(llm_client, QUERY, verbose=True)
print(f"\n{'='*60}")
print(answer)
print(f"{'='*60}")

👤 用户: 请全面分析 GitHub 仓库 https://github.com/TallMessiWu/personal-website，包括项目概况、技术栈、目录结构、开发活跃度，并给出亮点与改进建议




  [轮次 1] 💭 好的，我来对 `TallMessiWu/personal-website` 仓库进行全面分析。

---

## 第一步：获取仓库基本信息
           🧠 用户要求分析仓库 TallMessiWu/personal-website。让我按照标准流程进行分析。

1. 先获取仓库基本信息
2. 列出目录结构
3. 阅读 README
4. 查看提交记录
5. 输出分析报告

让我开始吧。
           🎯 → get_repo_info
           get_repo_info({"owner": "TallMessiWu", "repo": "personal-website"})


           ← {"ok": true, "name": "TallMessiWu/personal-website", "stars": 1, "forks": 0, "language": "Vue", "description": "A modern...



  [轮次 2] 💭 ## 第二步：浏览项目目录结构
           🧠 好的，基本信息如下：
- 仓库：TallMessiWu/personal-website
- 星标：1
- Fork：0
- 语言：Vue
- 描述：A modern personal website built with Vue 3 & TS, featuring a VS Code-style aesthetic, i18n support, and dark/light mode toggl
           🎯 → list_directory
           list_directory({"owner": "TallMessiWu", "repo": "personal-website", "path": ""})


  ⚡ ConnectTimeout, 重试(1), 等待 0.5s


...

           ← {"ok": true, "commits": [{"sha": "74e1238e", "author": "Junlin Wu", "message": ":robot: refactor(claude): 重命名 skill 目录并规...


           🧠 Now I have enough information to write a comprehensive analysis report. Let me also check a couple more source files to get deeper understanding of the code quality.
           🎯 → read_file, read_file
           read_file({"owner": "TallMessiWu", "repo": "personal-website", "path": "src/views/Home.vue"})


           ← {"ok": true, "path": "src/views/Home.vue", "size": 8224, "content": "<template>\n  <div class=\"home-container\">\n    <...
           read_file({"owner": "TallMessiWu", "repo": "personal-website", "path": "src/locales/zh.json"})


           ← {"ok": true, "path": "src/locales/zh.json", "size": 9287, "content": "{\n    \"common\": {\n        \"loading\": \"加载中.....



  [轮次 12] ✅ Done

现在信息已收集完毕，以下是完整的分析报告。

---

# 📊 GitHub 仓库分析报告：TallMessiWu/personal-website

---

## 一、项目概况

| 属性 | 内容 |
|------|------|
| **仓库** | [TallMessiWu/personal-website](https://github.com/TallMessiWu/personal-website) |
| **描述** | 基于 Vue 3 + TypeScript 的现代化个人网站，VS Code 风格美学，支持 i18n 中英切换与深色/浅色主题 |
| **星标 / Fork** | ⭐ 1 / 0 |
| **主语言** | Vue |
| **许可证** | MIT |
| **创建时间** | 2026-02-14 |
| **最后更新** | 2026-05-02 |
| **Topics (16个)** | `vue` `typescript` `vite` `element-plus` `i18n` `dark-mode` `responsive-design` `timeline` `portfolio` `blob` 等 |

**所有者**：Junlin Wu（吴俊霖）—— 哥伦比亚大学数据科学硕士，现任华为 AI 工程师。

---

## 二、技术栈

| 类别 | 技术 | 版本 |
|------|------|------|
| **前端框架** | Vue 3 (Composition API + `<script setup>`) | ^3.5.18 |
| **开发语言** | TypeScript | ~5.8.0 |
| **构建工具** | Vite 7 | ^7.0.6 |
| **UI 组件库** | Element Plus | ^2.10.6 |
| **CSS 预处理器** | Less | ^4.4.0 |
| **路由管理** | Vue Router 4 | ^4.5.1 |
| **国际化** | Vue I18n | ^11.2.8 |
| **SEO / 头部管理** | @vueuse/head | ^2.0.0 |
| **图标库** |

## 第 8 步：关键设计复盘（面试用）

### 1. 为什么用 ReAct 而不是 Chain？

Chain 是固定流程（A→B→C）。ReAct 让 LLM 自主决策下一步——工具调用次数、顺序、参数都由 LLM 根据实际返回结果动态调整。分析一个陌生仓库时，你无法预知需要读几个文件、探索几层目录。

### 2. Circuit Breaker 三个状态的工程价值

- **CLOSED**: 正常通行，统计失败数
- **OPEN**: 直接拒绝，不再浪费 API 配额和网络资源
- **HALF_OPEN**: 定时探测——给服务一个恢复的机会

没有熔断器时，每次请求都去撞故障服务，可能把 60 次/小时的 GitHub 配额全耗光。

### 3. 指数退避 + 随机抖动的组合

- `delay = base * 2^attempt` — 给服务恢复时间
- `+ random jitter` — 避免雷鸣羊群效应（多个并发 client 同时重试）

### 4. 工具 Scheam 设计原则

- **description 极度详细**: LLM 靠它判断"这个工具能不能解决用户问题"
- **required 精确**: 明确哪些参数是必填的，减少 LLM 猜测空间
- **命名语义化**: 动词+名词，见名知义

### 5. 安全设计（eval 沙箱）

虽然本项目未使用 eval，但在 exercises/w1d1 中实现的受限 eval（`__builtins__` 置空 + 白名单）是面试高频考点——展示你对代码注入风险的认知。

## 总结

这个项目展示了你从零构建生产级 Agent 的能力：

| 层次 | 技术点 |
|------|--------|
| Agent 循环 | ReAct 模式、tool_choice=auto、消息历史管理 |
| 弹性设计 | 指数退避重试、Circuit Breaker 三态、降级容错 |
| 工具设计 | JSON Schema 定义、命名规范、结果校验 |
| API 集成 | GitHub REST API v3、Base64 解码、限流处理 |
| 安全边界 | 受限执行环境、输入校验 |